In [1]:
# Binance price data cleaning
import pandas as pd

columns = ["date", "open", "high", "low", "close"]

file = "../data/raw/binance_with_volume.csv"

price = pd.read_csv(file,header=0,usecols=range(5),names=columns)

# print(price.head())

In [2]:
# Coinmetrics chain data cleaning

url = "https://raw.githubusercontent.com/coinmetrics/data/master/csv/btc.csv"

columns = [
    "time",
    "AdrActCnt",
    "TxCnt",
    "HashRate",
    "FeeTotNtv",
    "CapMVRVCur"
]

onchain = pd.read_csv(url, usecols=columns)

onchain = onchain.rename(columns={
    "time": "date",
    "AdrActCnt": "active_addresses",
    "TxCnt": "transaction_count",
    "HashRate": "hash_rate",
    "FeeTotNtv": "transaction_fees",
    "CapMVRVCur": "mvrv"
})

onchain = onchain[
    onchain["date"] >= "2017-12-01"
].copy()

print(onchain.isna().sum())

# print(onchain.head())

date                 0
active_addresses     1
mvrv                 1
transaction_fees     1
hash_rate            1
transaction_count    1
dtype: int64


In [3]:
from pathlib import Path

# Save local copies of necessary data
# Save local copy
output_dir = Path("../data/raw")
output_dir.mkdir(parents=True, exist_ok=True)

price.to_csv(output_dir / "price_raw.csv", index=True)

onchain.to_csv(output_dir / "onchain_raw.csv", index=True)


In [4]:
print(price.columns.tolist())
print(onchain.columns.tolist())


['date', 'open', 'high', 'low', 'close']
['date', 'active_addresses', 'mvrv', 'transaction_fees', 'hash_rate', 'transaction_count']


In [ ]:
import matplotlib as mpt
# Data cleaning

# Rename Coin Metrics 'time' to 'date
onchain = onchain.rename(columns={"time": "date"})

price["date"] = pd.to_datetime(
    price["date"],
    format="%Y-%m-%d",
    utc=True,
    errors="raise"
)

onchain["date"] = pd.to_datetime(
    onchain["date"],
    format="%Y-%m-%d",
    utc=True,
    errors="raise"
)

price = price.sort_values("date")
onchain = onchain.sort_values("date")

### Check missing values ###

# Determine how many missing in each col
print(price.isna().sum())
print(onchain.isna().sum())

# Determine how many dates are missing entirely.
expected_dates = pd.date_range(
    start=price["date"].min(),
    end=price["date"].max(),
    freq="D"
)

missing_dates = expected_dates.difference(price["date"])

print(missing_dates)

# Identify missing on chain observation
display(onchain[onchain.isna().any(axis=1)])

# Remove missing value (only 1 exists)
onchain = onchain.dropna().copy()

# Identify duplicates
print(price[price.duplicated("date", keep=False)])
print(onchain[onchain.duplicated("date", keep=False)])

# Remove identical observations
price = price.drop_duplicates()
onchain = onchain.drop_duplicates()

# Identify invalid values
invalid_prices = price[
    (price["open"] <= 0) | 
    (price["close"] <= 0) | 
    (price["high"] <= 0) |
    (price["low"] <= 0) |
    (price["high"] < price[["open", "close", "low"]].max(axis=1)) |
    (price["low"] > price[["open", "close", "low"]].min(axis=1))
]

print(invalid_prices)

# Check outliers
print(price.describe())
print(onchain.describe())

price.set_index("date")["close"].plot()

# Standardize reporting frequency by reindexing datasets to a complete daily calendar -- shifting by one row means shifting by one day
price = price.set_index("date").asfreq("D")
onchain = onchain.set_index("date").asfreq("D")

In [21]:
# Combine historical price and onchain data in 1 file and check it

combined = pd.merge(
    price,
    onchain,
    on="date",
    how="left",
    validate="one_to_one"
)

# Verify
print(combined.shape)

print(combined.head())

print(combined.isna().sum())

# Check duplicates
print(combined.columns)
print(combined.index.name)
print(combined.index.duplicated().sum())

print(len(price) == len(combined))


(3294, 9)
                              open     high      low    close  \
date                                                            
2017-11-13 00:00:00+00:00  5839.94  6697.47  5699.99  6465.99   
2017-11-14 00:00:00+00:00  6465.99  6684.98  6311.07  6574.99   
2017-11-15 00:00:00+00:00  6575.99  7298.00  6575.99  7240.06   
2017-11-16 00:00:00+00:00  7240.14  7940.00  7076.00  7864.50   
2017-11-17 00:00:00+00:00  7876.98  7989.00  7451.00  7699.19   

                           active_addresses  mvrv  transaction_fees  \
date                                                                  
2017-11-13 00:00:00+00:00               NaN   NaN               NaN   
2017-11-14 00:00:00+00:00               NaN   NaN               NaN   
2017-11-15 00:00:00+00:00               NaN   NaN               NaN   
2017-11-16 00:00:00+00:00               NaN   NaN               NaN   
2017-11-17 00:00:00+00:00               NaN   NaN               NaN   

                           hash_rate

In [ ]:
from pathlib import Path

# Save local copies of cleaned data
# Save local copy
output_dir = Path("../data/cleaned")
output_dir.mkdir(parents=True, exist_ok=True)

price.to_csv(output_dir / "price_cleaned.csv", index=True)

onchain.to_csv(output_dir / "onchain_cleaned.csv", index=True)

combined.to_csv(output_dir / "combined.csv",index=False)